# IR System — Evaluation
**Step 8:** Evaluate all retrieval models using MAP, Recall, P@10, nDCG@10.

Models evaluated: TF-IDF, BM25, Embedding, Hybrid Parallel (RRF), Hybrid Serial

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/ir_system_data'
import os, sys

if not os.path.exists('/content/ir-system'):
    !git clone https://github.com/ghazal-mohammad/ir-system.git /content/ir-system
else:
    !cd /content/ir-system && git pull
sys.path.insert(0, '/content/ir-system')

!pip install ir-datasets==0.5.9 -q
print('ready')

In [ ]:
import json
import ir_datasets
from services.evaluation_service import load_qrels, evaluate_run, print_results_table

# Load qrels
print('Loading qrels...')
ds1 = ir_datasets.load('clinicaltrials/2021/trec-ct-2021')
qrels1 = load_qrels(ds1)
print(f'CT2021 qrels: {len(qrels1)} queries with relevance judgements')

ds2 = ir_datasets.load('msmarco-passage/trec-dl-2019')
qrels2 = load_qrels(ds2)
print(f'MSMARCO qrels: {len(qrels2)} queries with relevance judgements')

In [ ]:
# Load all retrieval results for CT2021
def load_results(path):
    with open(path) as f:
        return json.load(f)

ct_tfidf = load_results(f'{SAVE_DIR}/ct2021_tfidf_results.json')
ct_bm25 = load_results(f'{SAVE_DIR}/ct2021_bm25_results.json')
ct_emb = load_results(f'{SAVE_DIR}/ct2021_embedding_results.json')
ct_hybrid_p = load_results(f'{SAVE_DIR}/ct2021_hybrid_parallel_results.json')
ct_hybrid_s = load_results(f'{SAVE_DIR}/ct2021_hybrid_serial_results.json')
print('CT2021 results loaded')

In [ ]:
# Evaluate all models on CT2021
print('\n=== CT2021 Evaluation ===')
ct_scores = {}
for name, run in [('TF-IDF', ct_tfidf), ('BM25', ct_bm25),
                   ('Embedding', ct_emb),
                   ('Hybrid-Parallel', ct_hybrid_p),
                   ('Hybrid-Serial', ct_hybrid_s)]:
    result = evaluate_run(run, qrels1)
    ct_scores[name] = result['aggregated']
    print(f'{name}: MAP={result["aggregated"]["MAP"]}')

print('\n--- CT2021 Full Comparison ---')
print_results_table(ct_scores)

# Save eval results
with open(f'{SAVE_DIR}/ct2021_eval_results.json', 'w') as f:
    json.dump(ct_scores, f, indent=2)
print('\nCT2021 eval saved')

In [ ]:
# Load all retrieval results for MSMARCO
ms_tfidf = load_results(f'{SAVE_DIR}/msmarco_tfidf_results.json')
ms_bm25 = load_results(f'{SAVE_DIR}/msmarco_bm25_results.json')
ms_emb = load_results(f'{SAVE_DIR}/msmarco_embedding_results.json')
ms_hybrid_p = load_results(f'{SAVE_DIR}/msmarco_hybrid_parallel_results.json')
ms_hybrid_s = load_results(f'{SAVE_DIR}/msmarco_hybrid_serial_results.json')
print('MSMARCO results loaded')

In [ ]:
# Evaluate all models on MSMARCO
print('\n=== MSMARCO Evaluation ===')
ms_scores = {}
for name, run in [('TF-IDF', ms_tfidf), ('BM25', ms_bm25),
                   ('Embedding', ms_emb),
                   ('Hybrid-Parallel', ms_hybrid_p),
                   ('Hybrid-Serial', ms_hybrid_s)]:
    result = evaluate_run(run, qrels2)
    ms_scores[name] = result['aggregated']
    print(f'{name}: MAP={result["aggregated"]["MAP"]}')

print('\n--- MSMARCO Full Comparison ---')
print_results_table(ms_scores)

with open(f'{SAVE_DIR}/msmarco_eval_results.json', 'w') as f:
    json.dump(ms_scores, f, indent=2)
print('\nMSMARCO eval saved')

print('\n=== Evaluation Complete ===')
print('Next: UI (app.py)')